In [2]:
from langchain.chat_models import init_chat_model
from rich import print as rprint
##统一使用init_chat_model（）完成模型的初始化
import os
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate


#1 读取，env文件中的配置信息，相关的环境变量以.env文件中的优先
load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#2 初始化大模型
model= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=3,
)
#3 声明1个函数（工具）
def get_weather(city:str):
    return f"{city}当天天气清理"
#4 将函数绑定在模型上
model_with_tools=model.bind_tools(
    [get_weather]
)
#5 调用模型
response=model_with_tools.invoke("北京的天气怎么样")
rprint(response)

AIMessage(
    content='',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 45,
            'prompt_tokens': 311,
            'total_tokens': 356,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 17,
                'rejected_prediction_tokens': None,
                'text_tokens': 45
            },
            'prompt_tokens_details': {
                'audio_tokens': None,
                'cache_write_tokens': None,
                'cached_tokens': 0,
                'image_tokens': None,
                'text_tokens': 311
            }
        },
        'model_provider': 'openai',
        'model_name': 'qwen3.8-max',
        'system_fingerprint': None,
        'id': 'chatcmpl-845c08eb-71ad-905e-a154-e1d6a5130efd',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--01a0a306-ca30-7251-bfb8-553856163696-0',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'city': '北京'},
            'id': 'call_0ffe2c4cb57640ba8e125df1',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 311,
        'output_tokens': 45,
        'total_tokens': 356,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 17}
    }
)

description说明

In [18]:
from langchain_core.utils.function_calling import convert_to_openai_tool
def get_weather(city:str="北京"):
#docstring中的描述有严格的要求，Args必须首字母大写，冒号必须是英文的，参数名称city，如果有描述，则函数声明的时候，指定的参数也必须说明其类型
#一旦参数设定了默认值，convert_to_openai_tool的输出结果中久没有required这个子元素了
    """
    查询城市的天气

    Args:
        city : 具体的城市名称，比如'北京'，'重庆'

    Returns:
        一串字符串，描述指定城市的天气情况

    """
    return f"{city}当天天气清理"
# #4 将函数绑定在模型上
# model_with_tools=model.bind_tools(
#     [get_weather]
# )
#5 调用模型
response=convert_to_openai_tool(get_weather)
rprint(response)

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {
            'properties': {
                'city': {
                    'default': '北京',
                    'description': "具体的城市名称，比如'北京'，'重庆'",
                    'type': 'string'
                }
            },
            'type': 'object'
        }
    }
}